In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



df =pd.read_csv('C:/Users/007/Desktop/nti/pima-indians-diabetes .csv')


In [3]:
print("Shape:",df.shape)
df.head(10)

Shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [6]:
df.nunique()

Pregnancies                  17
Glucose                     136
BloodPressure                47
SkinThickness                51
Insulin                     186
BMI                         248
DiabetesPedigreeFunction    517
Age                          52
Outcome                       2
dtype: int64

In [7]:
print("\n missing values  :  \n",df.isnull().sum()[df.isnull().sum()>0])


 missing values  :  
 Series([], dtype: int64)


In [8]:
print("\n number of duplicate rows: ", df.duplicated().sum())


 number of duplicate rows:  0


In [9]:
#The value is not zero medically

zero_counts = (df == 0).sum()
zero_percentages = ((df == 0).sum() / len(df) * 100).round(2)
print('\n zer0 values summary(count,percentage) \n')
zero_summary = pd.DataFrame(
    {
    'Zero_Count': zero_counts,
    'zero_percentage %': zero_percentages
}
)
print(zero_summary)



 zer0 values summary(count,percentage) 

                          Zero_Count  zero_percentage %
Pregnancies                      111              14.45
Glucose                            5               0.65
BloodPressure                     35               4.56
SkinThickness                    227              29.56
Insulin                          374              48.70
BMI                               11               1.43
DiabetesPedigreeFunction           0               0.00
Age                                0               0.00
Outcome                          500              65.10


In [10]:
#Replacing the zero values
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_clean = df.copy()

for col in zero_cols:
    df_clean[col] = df_clean.groupby('Outcome')[col].transform(
        lambda group: group.replace(0, group[group != 0].median())
    )

print('\n number of remaining zero\n')
print((df_clean[zero_cols] == 0).sum())


 number of remaining zero

Glucose          0
BloodPressure    0
SkinThickness    0
Insulin          0
BMI              0
dtype: int64


In [11]:
#skew_comparison
skew_comparison = pd.DataFrame({
    'Original Skewness': df.skew(),
    'Cleaned Skewness': df_clean.skew()
})
print('\n Skewness Comparison \n')
print(skew_comparison.round(2))


 Skewness Comparison 

                          Original Skewness  Cleaned Skewness
Pregnancies                            0.90              0.90
Glucose                                0.17              0.53
BloodPressure                         -1.84              0.14
SkinThickness                          0.11              0.82
Insulin                                2.27              3.03
BMI                                   -0.43              0.61
DiabetesPedigreeFunction               1.92              1.92
Age                                    1.13              1.13
Outcome                                0.64              0.64


In [12]:
#Outlier detection_IQR method 

df_final = df_clean.copy()
capped_summary = {}

for col in df_final.columns:
    if col == 'Outcome':
        continue
    Q1 = df_final[col].quantile(0.25)
    Q3 = df_final[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_capped = ((df_final[col] < lower_bound) | (df_final[col] > upper_bound)).sum()
    df_final[col] = df_final[col].clip(lower=lower_bound, upper=upper_bound)
    capped_summary[col] = n_capped

print('\n Outliers Capped Per Column (IQR bounds) \n')
print(pd.Series(capped_summary))

remaining = {}
for col in df_final.columns:
    if col == 'Outcome':
        continue
    Q1 = df_final[col].quantile(0.25)
    Q3 = df_final[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    remaining[col] = ((df_final[col] < lower_bound) | (df_final[col] > upper_bound)).sum()

print('\n number of remaining outliers after capping\n')
print(pd.Series(remaining))



 Outliers Capped Per Column (IQR bounds) 

Pregnancies                  4
Glucose                      0
BloodPressure               14
SkinThickness               87
Insulin                     51
BMI                          8
DiabetesPedigreeFunction    29
Age                          9
dtype: int64

 number of remaining outliers after capping

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
dtype: int64


In [13]:
#class distribution outcome
outcome_labels = df_clean['Outcome'].map({0: 'Non-Diabetic', 1: 'Diabetic' })

print('\n Outcome Class Distribution \n')
print(outcome_labels.value_counts())

print('\n Outcome Class Percentage (%) \n')
print((outcome_labels.value_counts(normalize=True) * 100).round(2))


 Outcome Class Distribution 

Outcome
Non-Diabetic    500
Diabetic        268
Name: count, dtype: int64

 Outcome Class Percentage (%) 

Outcome
Non-Diabetic    65.1
Diabetic        34.9
Name: proportion, dtype: float64


In [14]:
#correlation_with_outcome
correlation_with_outcome = df_clean.corr()['Outcome'].sort_values(ascending=False)
print("Correlation of each feature with Outcome:\n")
print(correlation_with_outcome.round(3))

Correlation of each feature with Outcome:

Outcome                     1.000
Glucose                     0.496
Insulin                     0.377
BMI                         0.316
SkinThickness               0.295
Age                         0.238
Pregnancies                 0.222
BloodPressure               0.174
DiabetesPedigreeFunction    0.174
Name: Outcome, dtype: float64
